# 🔮 명림당 사주 AI 개발 노트북

이 노트북은 **생년월일시를 입력받아 사주 팔자를 계산하고, AI로 맞춤 리딩을 생성**하는 전체 과정을 단계별로 설명합니다.

---

## 전체 파이프라인

```
생년월일시 입력
     ↓
① 사주 팔자 계산  (년주 / 월주 / 일주 / 시주)
     ↓
② 일간 특성 조회  (데이터베이스 lookup — AI 없음)
     ↓
③ 오행 분석      (木火土金水 분포, 용신 추론 — AI 없음)
     ↓
④ 프롬프트 구성  (분석 결과를 AI에 전달할 텍스트로 정리)
     ↓
⑤ Gemini AI 호출 (딱 1번 — 글쓰기만 담당)
     ↓
사주 리딩 결과 출력
```

> **핵심 설계 원칙**: AI는 계산하지 않습니다. 모든 사주 분석은 알고리즘으로 처리하고, AI는 그 결과를 자연스러운 한국어로 바꾸는 역할만 합니다.

---
## 0단계 — 필요한 패키지 설치

이 노트북은 외부 패키지를 최소화합니다.
- `requests` : Gemini API HTTP 호출용 (보통 기본 설치됨)
- `python-dotenv` : `.env` 파일에서 API 키를 안전하게 읽기 위해

In [ ]:
# 패키지 설치 (최초 1회만 실행)
# !pip install requests python-dotenv

import requests
import json
import os
from datetime import date
from dotenv import load_dotenv

# .env 파일에서 GEMINI_API_KEY 불러오기
load_dotenv()
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY', '')

print('✅ 패키지 로드 완료')
print(f'API 키 상태: {"설정됨 ✅" if GEMINI_API_KEY else "미설정 ❌ (.env 파일 확인 필요)"}')

---
## 1단계 — 사주의 기초 상수 정의

사주(四柱)는 **년·월·일·시** 네 기둥으로 이루어지며,  
각 기둥은 **천간(天干, 하늘의 기운)**과 **지지(地支, 땅의 기운)**의 조합입니다.

### 천간 (10개)
| 순서 | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 |
|------|---|---|---|---|---|---|---|---|---|---|
| 천간 | 갑 | 을 | 병 | 정 | 무 | 기 | 경 | 신 | 임 | 계 |
| 오행 | 木 | 木 | 火 | 火 | 土 | 土 | 金 | 金 | 水 | 水 |
| 음양 | 양 | 음 | 양 | 음 | 양 | 음 | 양 | 음 | 양 | 음 |

### 지지 (12개)
| 순서 | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 | 11 |
|------|---|---|---|---|---|---|---|---|---|---|----|----|---|
| 지지 | 자 | 축 | 인 | 묘 | 진 | 사 | 오 | 미 | 신 | 유 | 술 | 해 |
| 띠   | 쥐 | 소 | 호랑이 | 토끼 | 용 | 뱀 | 말 | 양 | 원숭이 | 닭 | 개 | 돼지 |
| 오행 | 水 | 土 | 木 | 木 | 土 | 火 | 火 | 土 | 金 | 金 | 土 | 水 |

In [ ]:
# ── 천간 (10개) ──────────────────────────────────────────────────
STEMS   = ['갑','을','병','정','무','기','경','신','임','계']
S_ELEM  = ['木','木','火','火','土','土','金','金','水','水']  # 천간 오행
S_YY    = ['양','음','양','음','양','음','양','음','양','음']  # 천간 음양

# ── 지지 (12개) ──────────────────────────────────────────────────
BRANCHES = ['자','축','인','묘','진','사','오','미','신','유','술','해']
B_ELEM   = ['水','土','木','木','土','火','火','土','金','金','土','水']  # 지지 오행
ANIMALS  = ['쥐','소','호랑이','토끼','용','뱀','말','양','원숭이','닭','개','돼지']  # 띠

# ── 오행 상생 관계: A가 B를 생(生)한다 ────────────────────────────
# 木→火→土→金→水→木  (나무가 불을 키우고, 불이 흙을 만들고...)
GENERATES_TO  = {'木':'火','火':'土','土':'金','金':'水','水':'木'}
GENERATED_BY  = {'火':'木','土':'火','金':'土','水':'金','木':'水'}  # 역방향

# ── 오행 상극 관계: A가 B를 극(剋)한다 ────────────────────────────
# 木→土→水→火→金→木
CONTROLS = {'木':'土','土':'水','水':'火','火':'金','金':'木'}

print('✅ 기초 상수 정의 완료')
print(f'천간: {" ".join(STEMS)}')
print(f'지지: {" ".join(BRANCHES)}')

---
## 2단계 — 년주(年柱) 계산

**년주**는 태어난 해의 기운을 나타내는 기둥입니다.

### 계산 공식
- **천간 인덱스** = `(태어난 해 - 4) % 10`
- **지지 인덱스** = `(태어난 해 - 4) % 12`

> **주의**: 사주에서 한 해의 시작은 양력 1월 1일이 아니라 **입춘(立春, 약 2월 4일)**입니다.  
> 2월 4일 이전에 태어났다면 전년도 기준으로 계산합니다.

In [ ]:
def get_year_pillar(year: int, month: int, day: int) -> dict:
    """
    년주(年柱) 계산
    - 입춘(약 2월 4일) 이전 출생자는 전년도 기준
    """
    # 입춘 이전이면 전년도를 기준으로
    saju_year = year - 1 if (month < 2 or (month == 2 and day < 4)) else year
    
    stem_idx   = (saju_year - 4) % 10
    branch_idx = (saju_year - 4) % 12
    
    return {
        'stem':       STEMS[stem_idx],
        'branch':     BRANCHES[branch_idx],
        'stem_elem':  S_ELEM[stem_idx],
        'branch_elem':B_ELEM[branch_idx],
        'yinyang':    S_YY[stem_idx],
        'animal':     ANIMALS[branch_idx],
        'char':       STEMS[stem_idx] + BRANCHES[branch_idx],
        '_si': stem_idx,  # 내부 계산용
    }

# ── 테스트 ──
yp = get_year_pillar(1993, 5, 20)
print(f'1993년 5월 20일 태어난 사람의 년주: {yp["char"]} ({yp["stem_elem"]}/{yp["branch_elem"]}, {yp["animal"]}띠)')

---
## 3단계 — 월주(月柱) 계산

**월주**는 태어난 달의 기운을 나타내는 기둥입니다.

### 핵심 개념: 절기(節氣)
양력 달과 사주의 달은 다릅니다. 사주에서 한 달의 시작은 각 달의 **절기 날**입니다.

| 양력 월 | 절기 | 시작일(근사값) | 사주 월 |
|---------|------|--------------|--------|
| 2월 | 입춘(立春) | 4일 | 인월(寅月) |
| 3월 | 경칩(驚蟄) | 6일 | 묘월(卯月) |
| 4월 | 청명(淸明) | 5일 | 진월(辰月) |
| ... | ... | ... | ... |

### 월 천간 계산 규칙
월의 천간은 **그 해의 천간**에 따라 시작점이 달라집니다:
- 갑년/기년 → 인월이 병(丙)으로 시작
- 을년/경년 → 인월이 무(戊)로 시작
- 병년/신년 → 인월이 경(庚)으로 시작
- 정년/임년 → 인월이 임(壬)으로 시작
- 무년/계년 → 인월이 갑(甲)으로 시작

In [ ]:
# 각 양력 달의 절기 시작일 (근사값)
# 인덱스 0 = 1월, 인덱스 11 = 12월
TERM_DAYS = [6, 4, 6, 5, 6, 6, 7, 7, 8, 8, 7, 7]
#            소한 입춘 경칩 청명 입하 망종 소서 입추 백로 한로 입동 대설

def get_month_pillar(year: int, month: int, day: int, year_stem_idx: int) -> dict:
    """
    월주(月柱) 계산
    - 절기 기준으로 사주 월(月) 결정
    - 해당 월의 절기 시작일 이전이면 전월 사주로 처리
    """
    term_day = TERM_DAYS[month - 1]
    
    # 절기 기준 사주 월 오프셋 (0=인월, 1=묘월, ...)
    if day >= term_day:
        month_offset = (month - 2 + 12) % 12  # 현재 달의 사주 월
    else:
        month_offset = (month - 3 + 12) % 12  # 절기 전이면 전월 사주
    
    # 지지: 인(寅)=2 부터 시작하므로 +2
    branch_idx = (month_offset + 2) % 12
    
    # 천간: 해당 연도 천간 그룹에서 기준 천간 결정
    # 갑(0)/기(5)→병(2), 을(1)/경(6)→무(4), 병(2)/신(7)→경(6), 정(3)/임(8)→임(8), 무(4)/계(9)→갑(0)
    year_group = (year_stem_idx // 2) % 5
    base_stem  = [2, 4, 6, 8, 0][year_group]
    stem_idx   = (base_stem + month_offset) % 10
    
    return {
        'stem':        STEMS[stem_idx],
        'branch':      BRANCHES[branch_idx],
        'stem_elem':   S_ELEM[stem_idx],
        'branch_elem': B_ELEM[branch_idx],
        'char':        STEMS[stem_idx] + BRANCHES[branch_idx],
    }

# ── 테스트 ──
yp = get_year_pillar(1993, 5, 20)
mp = get_month_pillar(1993, 5, 20, yp['_si'])
print(f'1993년 5월 20일의 월주: {mp["char"]} ({mp["stem_elem"]}/{mp["branch_elem"]})')

---
## 4단계 — 일주(日柱) 계산

**일주**는 사주에서 가장 중요한 기둥입니다.  
일주의 천간을 **일간(日干)**이라 하며, 이것이 바로 **나 자신의 본질**을 나타냅니다.

### 계산 방법: 기준일 방식
60갑자는 60일마다 한 바퀴를 돕니다.  
**기준일**: `2000년 1월 1일 = 경진(庚辰) = 인덱스 16`

```
일주 인덱스 = (기준일로부터 경과 일수 + 16) % 60
천간 인덱스 = 일주 인덱스 % 10
지지 인덱스 = 일주 인덱스 % 12
```

> **왜 경진이 기준인가?** 2000년 1월 1일의 일진이 경진임이 역학 문헌에서 확인됩니다.

In [ ]:
def get_day_pillar(year: int, month: int, day: int) -> dict:
    """
    일주(日柱) 계산
    - 기준일: 2000-01-01 = 경진(庚辰) = 60갑자 인덱스 16
    """
    ref    = date(2000, 1, 1)
    target = date(year, month, day)
    delta  = (target - ref).days  # 기준일로부터 경과 일수
    
    idx        = (delta + 16) % 60
    stem_idx   = idx % 10
    branch_idx = idx % 12
    
    return {
        'stem':        STEMS[stem_idx],
        'branch':      BRANCHES[branch_idx],
        'stem_elem':   S_ELEM[stem_idx],
        'branch_elem': B_ELEM[branch_idx],
        'yinyang':     S_YY[stem_idx],
        'char':        STEMS[stem_idx] + BRANCHES[branch_idx],
        '_si': stem_idx,  # 내부 계산용
    }

# ── 테스트 ──
dp = get_day_pillar(1993, 5, 20)
print(f'1993년 5월 20일의 일주: {dp["char"]} (일간: {dp["stem"]} — 이 사람의 본질)')

---
## 5단계 — 시주(時柱) 계산

**시주**는 태어난 시각의 기운을 나타내는 기둥입니다.

### 한국 전통 시간 단위
하루를 12개 시(時)로 나눕니다. 각 시는 현대 시간으로 2시간입니다.

| 시 | 시간대 | 지지 |
|----|--------|------|
| 자시(子時) | 23:00~01:00 | 자(子) |
| 축시(丑時) | 01:00~03:00 | 축(丑) |
| 인시(寅時) | 03:00~05:00 | 인(寅) |
| ... | ... | ... |
| 해시(亥時) | 21:00~23:00 | 해(亥) |

### 시 천간 계산
시의 천간은 **일간(日干)**에 따라 결정됩니다.

In [ ]:
# 한국 전통 시(時) → 현대 시각 범위
HOUR_NAMES = [
    ('자시', '23:00~01:00'), ('축시', '01:00~03:00'), ('인시', '03:00~05:00'),
    ('묘시', '05:00~07:00'), ('진시', '07:00~09:00'), ('사시', '09:00~11:00'),
    ('오시', '11:00~13:00'), ('미시', '13:00~15:00'), ('신시', '15:00~17:00'),
    ('유시', '17:00~19:00'), ('술시', '19:00~21:00'), ('해시', '21:00~23:00'),
]

def get_hour_pillar(hour: int, day_stem_idx: int) -> dict:
    """
    시주(時柱) 계산
    - hour: 0~23 (24시간 형식)
    - 자시(23:00~01:00)는 당일 기준 처리
    """
    # 지지 결정: 23시는 자시(0), 나머지는 (hour+1)//2
    branch_idx = 0 if hour == 23 else ((hour + 1) // 2) % 12
    
    # 천간 기준 (일간 그룹에 따라 자시 천간이 달라짐)
    # 갑(0)/기(5)→갑(0), 을(1)/경(6)→병(2), 병(2)/신(7)→무(4), 정(3)/임(8)→경(6), 무(4)/계(9)→임(8)
    base_stem  = [0, 2, 4, 6, 8][day_stem_idx % 5]
    stem_idx   = (base_stem + branch_idx) % 10
    
    return {
        'stem':        STEMS[stem_idx],
        'branch':      BRANCHES[branch_idx],
        'stem_elem':   S_ELEM[stem_idx],
        'branch_elem': B_ELEM[branch_idx],
        'char':        STEMS[stem_idx] + BRANCHES[branch_idx],
        'hour_name':   HOUR_NAMES[branch_idx][0],
    }

# ── 테스트 ──
hp = get_hour_pillar(14, dp['_si'])  # 오후 2시 (미시)
print(f'오후 2시(14시)의 시주: {hp["char"]} ({hp["hour_name"]})')

---
## 6단계 — 사주 팔자(四柱八字) 통합 계산

지금까지 만든 4개의 함수를 하나로 합칩니다.  
**사주 팔자** = 4개의 기둥 × 각 2글자(천간+지지) = 총 8글자

In [ ]:
def calc_saju(year: int, month: int, day: int, hour: int) -> dict:
    """
    사주 팔자 전체 계산
    Returns: 년주/월주/일주/시주 + 띠 + 나이 정보
    """
    year_p  = get_year_pillar(year, month, day)
    month_p = get_month_pillar(year, month, day, year_p['_si'])
    day_p   = get_day_pillar(year, month, day)
    hour_p  = get_hour_pillar(hour, day_p['_si'])
    
    return {
        'year':   year_p,
        'month':  month_p,
        'day':    day_p,
        'hour':   hour_p,
        'animal': year_p['animal'],
    }

def print_saju(pillars: dict, name: str = ''):
    """사주 팔자를 보기 좋게 출력"""
    sep = '=' * 42
    print('\n' + sep)
    if name:
        print(f'  {name}님의 사주 팔자')
    print(sep)
    labels = ['년주(年柱)', '월주(月柱)', '일주(日柱)', '시주(時柱)']
    keys   = ['year', 'month', 'day', 'hour']
    for lbl, key in zip(labels, keys):
        p = pillars[key]
        marker = ' ← 나의 본질 (일간)' if key == 'day' else ''
        print(f'  {lbl}: {p["char"]}  ({p["stem_elem"]}/{p["branch_elem"]}){marker}')
    print(f'  띠: {pillars["animal"]}띠')
    print(sep + '\n')

# ── 테스트: 직접 바꿔보세요! ──────────────────────────────────────
TEST_NAME  = '홍길동'
TEST_YEAR  = 1993   # ← 여기를 바꿔보세요
TEST_MONTH = 5
TEST_DAY   = 20
TEST_HOUR  = 14     # 24시간 형식 (모르면 12 입력)

pillars = calc_saju(TEST_YEAR, TEST_MONTH, TEST_DAY, TEST_HOUR)
print_saju(pillars, TEST_NAME)

---
## 7단계 — 일간(日干) 특성 데이터베이스

일간은 10가지(갑~계)이며, 각각의 성격·직업·연애·재물 특성이 명리학에서 정해져 있습니다.  
**이 데이터는 AI 없이 코드에 직접 저장됩니다.**

---
### ✏️ 수정 가이드
아래 딕셔너리의 각 항목을 자유롭게 수정하면 리딩 내용이 바뀝니다.
- `personality` : 성격 설명 (이게 가장 중요!)
- `strengths` : 강점
- `weaknesses` : 약점
- `career` : 잘 맞는 직업
- `love` : 연애 스타일
- `money` : 재물 성향
- `advice` : 핵심 조언

In [ ]:
# ── 일간별 특성 데이터베이스 ─────────────────────────────────────────────
# 이 딕셔너리를 수정하면 AI에게 전달되는 분석 내용이 바뀝니다

ILGAN = {
    '갑': {
        'nature': '양목(陽木)', 'symbol': '우뚝 선 소나무',
        'personality': '리더십이 강하고 목표를 향해 직진하는 기질. 정의감이 넘치고 한번 마음먹으면 끝까지 밀어붙이는 추진력이 있음.',
        'strengths': '추진력, 리더십, 정직함, 책임감, 개척정신',
        'weaknesses': '고집, 융통성 부족, 직선적이라 상처를 줄 수 있음',
        'career': '경영·창업, 법조, 교육, 건설, 의료',
        'love': '한 사람에게 깊고 충실하나 자존심이 강해 먼저 사과하기 어려움. 부드럽고 포용력 있는 상대와 잘 맞음.',
        'money': '꾸준한 노력으로 재물을 쌓는 스타일. 성실한 사업이나 직장에서 재물운이 옴.',
        'advice': '때로는 한 발 물러서는 유연함도 힘입니다.',
    },
    '을': {
        'nature': '음목(陰木)', 'symbol': '유연하게 뻗는 덩굴',
        'personality': '부드럽고 유연하지만 내면은 질긴 생명력을 가짐. 예술적 감각과 섬세함이 뛰어나며 사람들과 잘 어울림.',
        'strengths': '유연성, 친화력, 예술적 감각, 적응력, 끈기',
        'weaknesses': '우유부단, 의존적 성향, 자기주장이 약함',
        'career': '예술·디자인, 미용·뷰티, 교육, 서비스, 의료',
        'love': '감성적이고 로맨틱한 연애를 원함. 안정감 있고 든든한 상대에게 끌림.',
        'money': '재물은 여러 경로에서 조금씩 모이는 형태. 저축하는 습관이 중요.',
        'advice': '스스로의 의견을 더 강하게 표현해도 됩니다.',
    },
    '병': {
        'nature': '양화(陽火)', 'symbol': '뜨겁게 타오르는 태양',
        'personality': '밝고 외향적이며 어디서든 존재감이 넘침. 열정이 강하고 카리스마가 있으나 감정 기복이 크고 충동적인 면이 있음.',
        'strengths': '카리스마, 열정, 사교성, 표현력, 긍정적 에너지',
        'weaknesses': '충동성, 감정 기복, 지속력 부족, 인정욕구가 강함',
        'career': '연예·방송, 영업·마케팅, 정치, 강연, 요식업',
        'love': '열정적이고 표현이 풍부한 연애. 빨리 불태우고 빨리 식는 경향.',
        'money': '돈이 들어오면 빠르게 쓰는 성향. 계획적인 저축이 중요.',
        'advice': '뜨거운 열정만큼 꾸준함도 갖추면 무서운 사람이 됩니다.',
    },
    '정': {
        'nature': '음화(陰火)', 'symbol': '은은하게 타오르는 촛불',
        'personality': '섬세하고 집중력이 강하며 한번 빠지면 깊이 파고드는 성격. 겉으로는 차분해 보이나 내면은 뜨거운 열정을 품고 있음.',
        'strengths': '집중력, 섬세함, 직관력, 예술성, 깊이 있는 사고',
        'weaknesses': '과도한 집착, 완벽주의, 내성적인 면',
        'career': '예술·음악, 연구·학문, IT·기술, 종교·철학, 의료',
        'love': '조용하지만 깊고 진지한 연애를 원함. 감정 상처를 오래 품는 경향.',
        'money': '전문성을 바탕으로 한 수입이 주된 재물 루트.',
        'advice': '완벽하지 않아도 괜찮습니다. 당신의 내면 빛은 이미 충분합니다.',
    },
    '무': {
        'nature': '양토(陽土)', 'symbol': '묵직하고 넓은 대지',
        'personality': '안정적이고 신뢰감이 높으며 주변을 든든하게 지켜줌. 대범하고 포용력이 있으나 변화를 싫어하는 고집이 있음.',
        'strengths': '안정감, 신뢰감, 포용력, 인내심, 실행력',
        'weaknesses': '변화 거부, 고집, 느린 결정',
        'career': '금융·보험, 부동산, 경영관리, 농업·식품, 공무원',
        'love': '천천히 깊어지는 연애 스타일. 한번 맺은 인연을 소중히 여기고 오래 감.',
        'money': '부동산·안정 자산에서 재물운이 강함. 꾸준히 축적하는 방식이 맞음.',
        'advice': '변화를 두려워하지 마세요. 안정성이 발판이 되어야지 족쇄가 되면 안 됩니다.',
    },
    '기': {
        'nature': '음토(陰土)', 'symbol': '세심하게 가꾸는 비옥한 밭',
        'personality': '꼼꼼하고 세심하며 현실적인 판단력이 뛰어남. 배려심이 깊고 실용적이나 걱정이 많고 소심한 면이 있음.',
        'strengths': '세심함, 현실감각, 실무능력, 배려심, 꼼꼼함',
        'weaknesses': '걱정이 많음, 소심함, 확장성 부족',
        'career': '교육, 상담·복지, 의료·간호, 행정·사무, 식품',
        'love': '상대를 세심하게 챙기는 연애 스타일. 불안감이 많아 확인을 자주 원함.',
        'money': '알뜰하게 모으는 스타일. 재물이 조금씩 꾸준히 쌓임.',
        'advice': '걱정은 미래를 바꾸지 못합니다. 준비됐다면 과감하게 나아가도 됩니다.',
    },
    '경': {
        'nature': '양금(陽金)', 'symbol': '결단력 있는 단단한 쇠',
        'personality': '결단력이 강하고 원칙을 중시하며 의리가 있음. 카리스마와 리더십이 있으나 독선적이 될 수 있음.',
        'strengths': '결단력, 의리, 실행력, 리더십, 카리스마',
        'weaknesses': '독선, 완고함, 감정표현이 서툼',
        'career': '군인·경찰, 법조, 스포츠, 제조·기술, 경영',
        'love': '책임감 있고 의리 있는 파트너. 행동으로 사랑을 표현하나 감정 표현이 서툼.',
        'money': '승부 기질이 있어 사업·투자에서 큰 재물 가능. 한방에 무너지는 위험도 있으니 분산 필요.',
        'advice': '날카로운 칼도 칼집이 있을 때 더 빛납니다. 부드러움이 당신을 더 강하게 합니다.',
    },
    '신': {
        'nature': '음금(陰金)', 'symbol': '아름다움을 추구하는 보석',
        'personality': '예민하고 섬세하며 미적 감각이 뛰어남. 완벽주의 성향이 있고 자신과 주변에 높은 기준을 세움.',
        'strengths': '섬세함, 미적감각, 완벽주의, 직관력, 분석력',
        'weaknesses': '예민함, 상처를 잘 받음, 비판적 성향',
        'career': '예술·패션·뷰티, 의료·치과, 디자인, 법조, IT',
        'love': '이상형이 높고 까다로운 연애 기준을 가짐. 상처를 잘 받고 오래 기억함.',
        'money': '섬세한 분석력으로 재물 관리를 잘함. 패션·뷰티·예술 분야에서 수입이 좋음.',
        'advice': '완벽하지 않아도 충분히 사랑받을 자격이 있습니다. 스스로를 너그럽게 대해주세요.',
    },
    '임': {
        'nature': '양수(陽水)', 'symbol': '깊고 넓은 바다',
        'personality': '지혜롭고 통찰력이 있으며 큰 그림을 봄. 포용력이 넓고 변화에 강하나 결정적일 때 우유부단해지기도 함.',
        'strengths': '지혜, 포용력, 통찰력, 유연성, 창의성',
        'weaknesses': '우유부단, 비밀이 많음, 감정 기복',
        'career': '연구·학문, IT·인터넷, 심리·상담, 예술, 무역·외교',
        'love': '깊고 신비로운 매력으로 이성을 끌어당김. 자유로운 관계를 선호하며 속박을 답답해함.',
        'money': '지식·정보·창의성으로 돈을 버는 타입. 여러 수입원이 유리.',
        'advice': '당신의 깊이는 남들이 쉽게 따라올 수 없는 무기입니다. 그 지혜를 행동으로 연결하세요.',
    },
    '계': {
        'nature': '음수(陰水)', 'symbol': '섬세하고 깊은 이슬',
        'personality': '감수성이 풍부하고 직관력이 뛰어남. 내성적이나 마음을 열면 깊은 유대감을 형성. 타인의 감정을 잘 캐치함.',
        'strengths': '감수성, 직관력, 섬세함, 공감능력, 예술성',
        'weaknesses': '내성적, 감정 상처가 깊음, 우울감 경향',
        'career': '예술·문학, 심리·상담, 의료, 교육, 종교·철학',
        'love': '감성적이고 로맨틱한 사랑을 꿈꿈. 공감해주는 따뜻한 상대에게 깊이 빠짐.',
        'money': '예술·감성 분야에서 재능을 수입으로 연결하는 것이 유리. 감정적 소비를 조심.',
        'advice': '당신의 감수성은 세상이 미처 보지 못한 것을 보는 힘입니다.',
    },
}

# ── 테스트 ──
day_stem = pillars['day']['stem']
traits = ILGAN[day_stem]
print(f'일간 {day_stem} ({traits["nature"]}) — {traits["symbol"]}')
print(f'성격: {traits["personality"]}')
print(f'강점: {traits["strengths"]}')
print(f'약점: {traits["weaknesses"]}')

---
## 8단계 — 오행(五行) 분석

사주 팔자의 8글자(천간 4개 + 지지 4개) 각각의 오행을 세어  
**어떤 기운이 많고 적은지** 파악합니다.

### 분석 항목
- **오행 분포**: 木·火·土·金·水 각각 몇 개인지 카운트
- **강한 오행**: 가장 많은 기운 → 성격에 강하게 드러남
- **약한 오행**: 가장 적은 기운 → 부족한 면, 보완이 필요
- **용신(用神)**: 약한 기운을 도와주는 기운 (상생 원리 적용)

In [ ]:
def analyze_elements(pillars: dict) -> dict:
    """
    오행 분석
    - 4개 기둥 × (천간 오행 + 지지 오행) = 8개 데이터 포인트
    - 용신: 약한 오행을 생(生)하는 오행
    """
    count = {'木': 0, '火': 0, '土': 0, '金': 0, '水': 0}
    
    for key in ['year', 'month', 'day', 'hour']:
        p = pillars[key]
        count[p['stem_elem']]  += 1  # 천간 오행
        count[p['branch_elem']] += 1  # 지지 오행
    
    sorted_elems = sorted(count.items(), key=lambda x: x[1], reverse=True)
    dominant = sorted_elems[0][0]   # 가장 많은 오행
    weak     = sorted_elems[-1][0]  # 가장 적은 오행
    
    # 용신: 약한 오행을 생(生)해주는 오행
    yongsin = GENERATED_BY.get(weak, dominant)
    
    return {
        'count':    count,
        'dominant': dominant,
        'weak':     weak,
        'yongsin':  yongsin,
    }

def print_elements(elem: dict):
    """오행 분석 결과 출력"""
    print('오행 분포:')
    for e, cnt in elem['count'].items():
        bar = '█' * cnt + '░' * (8 - cnt)
        print(f'  {e} {bar} {cnt}개')
    print(f'\n강한 기운: {elem["dominant"]}  |  약한 기운: {elem["weak"]}  |  용신: {elem["yongsin"]}')

# ── 테스트 ──
elem = analyze_elements(pillars)
print_elements(elem)

---
## 9단계 — AI 프롬프트 구성

지금까지 계산한 모든 결과를 AI에 전달할 텍스트로 정리합니다.  
AI는 이 데이터를 **자연스러운 한국어 리딩으로 변환**하는 역할만 합니다.

### 프롬프트 설계 원칙
1. 이미 계산된 데이터를 그대로 제공 (AI가 새로 계산하지 않도록)
2. 출력 형식(JSON)을 명확히 지정
3. 어조와 길이를 구체적으로 가이드

> ✏️ **수정 포인트**: 아래 prompt 템플릿의 각 섹션 설명을 바꾸면 리딩 내용이 달라집니다!

In [ ]:
def build_prompt(name: str, gender: str, year: int, month: int, day: int,
                 pillars: dict, elem: dict, traits: dict,
                 hour_unknown: bool = False) -> str:
    """
    Gemini에게 전달할 프롬프트 생성
    - 사전 계산된 모든 데이터를 구조화해서 전달
    - AI는 글쓰기만 담당
    """
    age      = date.today().year - year + 1
    cur_year = date.today().year
    elem_str = ' '.join(f'{k}{v}개' for k, v in elem['count'].items())
    
    analysis = f"""
[사전 분석 완료 데이터 — AI는 이 데이터를 바탕으로 글만 써주세요]

의뢰인: {name} ({gender}, {year}년생 {pillars['animal']}띠, 현재 {age}세)
일간:   {pillars['day']['stem']} ({traits['nature']}) — {traits['symbol']}

사주 팔자:
  년주 {pillars['year']['char']} / 월주 {pillars['month']['char']} / 일주 {pillars['day']['char']} / 시주 {pillars['hour']['char']}{' (시간 미상)' if hour_unknown else ''}

오행 분포: {elem_str}
  강한 오행: {elem['dominant']} / 약한 오행: {elem['weak']} / 용신: {elem['yongsin']}

성격:    {traits['personality']}
강점:    {traits['strengths']}
약점:    {traits['weaknesses']}
직업:    {traits['career']}
연애:    {traits['love']}
재물:    {traits['money']}
조언:    {traits['advice']}
""".strip()

    # ── 프롬프트 템플릿 (이 부분을 수정하면 리딩 스타일이 바뀝니다) ──
    prompt = f"""당신은 사주 명리학 글쓰기 전문가입니다.
아래 사전 분석 데이터를 바탕으로, {name}님을 위한 따뜻하고 신비로운 사주 리딩을 JSON으로 작성해주세요.
데이터를 그대로 읽지 말고, 자연스럽고 감성적인 한국어 문장으로 풀어쓰세요.

{analysis}

마크다운·코드블록 없이 순수 JSON만 출력하세요:
{{
  "intro": "{name}님에게 건네는 첫 마디. 띠와 일간의 이미지를 엮어 2-3문장으로.",
  "sections": [
    {{"id":"personality","title":"타고난 기질과 성품","summary":"한 줄 핵심","content":"4-5문장 상세 리딩"}},
    {{"id":"love","title":"연애와 인연","summary":"한 줄 핵심","content":"4-5문장"}},
    {{"id":"money","title":"재물과 돈","summary":"한 줄 핵심","content":"4-5문장"}},
    {{"id":"career","title":"직업과 커리어","summary":"한 줄 핵심","content":"4-5문장"}},
    {{"id":"thisyear","title":"{cur_year}년 운세","summary":"한 줄 핵심","content":"4-5문장"}},
    {{"id":"flow","title":"인생의 큰 흐름","summary":"한 줄 핵심","content":"4-5문장"}}
  ]
}}"""
    return prompt

# ── 프롬프트 미리보기 ──
prompt = build_prompt(
    name=TEST_NAME, gender='남성',
    year=TEST_YEAR, month=TEST_MONTH, day=TEST_DAY,
    pillars=pillars, elem=elem, traits=ILGAN[pillars['day']['stem']]
)
print('생성된 프롬프트 (앞 500자):')
print('─' * 50)
print(prompt[:500] + '...')
print('─' * 50)
print(f'\n총 프롬프트 길이: {len(prompt)}자')

---
## 10단계 — Gemini API 호출

준비된 프롬프트를 Gemini에 전송하고 결과를 받습니다.  
**SDK 없이 HTTP 요청 직접 사용** — 더 안정적이고 어떤 환경에서도 동작합니다.

### 사용 모델
- `gemini-2.0-flash` : 무료 티어 사용 가능, 빠른 응답

### 설정값 설명
- `maxOutputTokens` : 응답 최대 길이 (높일수록 더 길게 씀, 하지만 더 느림)
- `temperature` : 창의성 수준 (0=딱딱함, 1=창의적, 0.85 권장)

In [ ]:
def call_gemini(prompt: str, api_key: str,
                max_tokens: int = 2500,
                temperature: float = 0.85) -> str:
    """
    Gemini API 호출 (HTTP 직접 요청)
    Returns: AI가 생성한 텍스트
    """
    url = (
        'https://generativelanguage.googleapis.com/v1beta/models/'
        f'gemini-2.0-flash:generateContent?key={api_key}'
    )
    payload = {
        'contents': [{'parts': [{'text': prompt}]}],
        'generationConfig': {
            'maxOutputTokens': max_tokens,
            'temperature': temperature,
        }
    }
    
    response = requests.post(url, json=payload, timeout=60)
    
    if response.status_code != 200:
        raise Exception(f'API 오류 {response.status_code}: {response.text[:300]}')
    
    result = response.json()
    text   = result['candidates'][0]['content']['parts'][0]['text']
    return text


def clean_json(text: str) -> str:
    """AI 응답에서 순수 JSON만 추출 (코드블록 등 제거)"""
    import re
    # ```json ... ``` 형태 처리
    match = re.search(r'```(?:json)?\s*([\s\S]*?)```', text)
    if match:
        return match.group(1).strip()
    return text.strip()


print('✅ Gemini API 함수 준비 완료')
print(f'API 키: {"설정됨 ✅" if GEMINI_API_KEY else "미설정 ❌"}')

---
## 11단계 — 전체 파이프라인 실행

지금까지 만든 모든 함수를 하나로 합쳐 실행합니다.

### ✏️ 여기에서 테스트할 정보를 입력하세요!

In [ ]:
# ════════════════════════════════════════════════════════
#  여기를 수정해서 원하는 사람의 사주를 볼 수 있습니다!
# ════════════════════════════════════════════════════════
INPUT_NAME   = '홍길동'   # ← 이름
INPUT_GENDER = '남성'     # ← '남성' 또는 '여성'
INPUT_YEAR   = 1993       # ← 태어난 년도
INPUT_MONTH  = 5          # ← 태어난 월
INPUT_DAY    = 20         # ← 태어난 일
INPUT_HOUR   = 14         # ← 태어난 시각 (24시간, 모르면 12)
HOUR_UNKNOWN = False      # ← 시각 모르면 True
# ════════════════════════════════════════════════════════

print('사주 리딩을 생성합니다...')
print('=' * 50)

# ① 사주 팔자 계산
print('① 사주 팔자 계산 중...')
pillars = calc_saju(INPUT_YEAR, INPUT_MONTH, INPUT_DAY, INPUT_HOUR)
print_saju(pillars, INPUT_NAME)

# ② 일간 특성 조회
print('② 일간 특성 조회 중...')
traits = ILGAN[pillars['day']['stem']]
print(f'   일간: {pillars["day"]["stem"]} — {traits["symbol"]}')

# ③ 오행 분석
print('③ 오행 분석 중...')
elem = analyze_elements(pillars)
print_elements(elem)

# ④ 프롬프트 구성
print('④ AI 프롬프트 구성 중...')
prompt = build_prompt(
    name=INPUT_NAME, gender=INPUT_GENDER,
    year=INPUT_YEAR, month=INPUT_MONTH, day=INPUT_DAY,
    pillars=pillars, elem=elem, traits=traits,
    hour_unknown=HOUR_UNKNOWN
)
print(f'   프롬프트 길이: {len(prompt)}자')

# ⑤ Gemini API 호출 (딱 1번)
print('⑤ Gemini AI 호출 중... (약 10~20초 소요)')
raw_response = call_gemini(prompt, GEMINI_API_KEY)
print('   AI 응답 수신 완료 ✅')

---
## 12단계 — 결과 출력

AI가 생성한 JSON을 파싱해서 보기 좋게 출력합니다.

In [ ]:
# JSON 파싱
try:
    reading = json.loads(clean_json(raw_response))
    
    sep = '\u2550' * 55  # ═ 문자
    line = '\u2500' * 55  # ─ 문자
    
    print('\n' + sep)
    print(f'  명림당 \u2014 {INPUT_NAME}님의 사주 리딩')
    print(sep)
    
    # 인트로
    print(f'\n{reading["intro"]}\n')
    print(line)
    
    # 각 섹션 출력
    for section in reading['sections']:
        print(f'\n\u258c {section["title"]}')
        print(f'  \u2192 {section["summary"]}')
        print(f'\n{section["content"]}')
        print(line)

except json.JSONDecodeError:
    print('JSON 파싱 실패 — 원본 응답:')
    print(raw_response)

---
## 13단계 — 웹 서버로 저장

노트북에서 검증한 알고리즘을 `server.js`에 반영하려면 아래 함수를 참고하세요.

### 노트북 ↔ 서버 매핑

| 노트북 함수 | server.js 함수 |
|------------|---------------|
| `calc_saju()` | `calcSaju()` |
| `analyze_elements()` | `analyzeElements()` |
| `ILGAN` 딕셔너리 | `ILGAN` 객체 |
| `build_prompt()` | 프롬프트 템플릿 리터럴 |
| `call_gemini()` | `fetch(Gemini URL)` |

In [ ]:
# ── 결과 요약 출력 ───────────────────────────────────────────────────────
print('사주 분석 요약')
print('=' * 40)
print(f'이름  : {INPUT_NAME} ({INPUT_GENDER})')
print(f'생년월일: {INPUT_YEAR}년 {INPUT_MONTH}월 {INPUT_DAY}일')
print(f'띠    : {pillars["animal"]}띠')
print(f'일간  : {pillars["day"]["stem"]} ({traits["nature"]}) — {traits["symbol"]}')
print(f'사주  : {pillars["year"]["char"]} {pillars["month"]["char"]} {pillars["day"]["char"]} {pillars["hour"]["char"]}')
print(f'강한기운: {elem["dominant"]}  |  약한기운: {elem["weak"]}  |  용신: {elem["yongsin"]}')
print('=' * 40)
print('\n✅ 완료! 위 결과를 바탕으로 ILGAN 데이터를 수정하면 리딩이 달라집니다.')